# 02 · Descriptive analysis

Sample, prevalence of out-of-school children, completeness of each predictor, and its
association with attending school.

**Input:** `data/analysis_data.parquet`. **Output:** `results/descriptives/*.csv`,
`results/tables/tableD1_predictors.csv`, `results/tables/table4_disability_by_head_education.csv`.

1. Setup
2. Sample and out-of-school prevalence
3. Attendance by group
4. Missing values
5. Association with attendance
6. School-dependent predictors
7. Table D1: predictors used
8. Subgroups: age, sex, disability (Table 4)

## Step 1 · Setup

In [1]:
import sys
sys.path.insert(0, "..")
import src  # noqa: F401  - loads TensorFlow before pandas (required on Windows)

import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 80)

from scipy.stats import chi2_contingency

from src.config import (DATA_DIR, FEATURE_DOMAINS, FEATURE_LEVEL, FEATURE_NAMES, FEATURES,
                        LABEL, MAX_MISSING_SHARE, RESULTS_DIR, SCHOOL_DEPENDENT, TARGET)

OUT = RESULTS_DIR / "descriptives"
TABLES = RESULTS_DIR / "tables"
OUT.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)
data = pd.read_parquet(DATA_DIR / "analysis_data.parquet")
data["district"] = data["district_key"].map(LABEL)
ORDER = list(LABEL.values())
data.shape

(30592, 65)

## Step 2 · Sample and out-of-school prevalence
Survey-weighted figures use the PSLM sampling weights.

In [2]:
def weighted_share_out(d):
    return (1 - np.average(d[TARGET], weights=d["weights"])) * 100

g = data.groupby("district")
sample = pd.DataFrame({
    "children": g.size(),
    "households": g["hhcode"].nunique(),
    "PSUs": g["psu"].nunique(),
    "out of school (n)": g[TARGET].apply(lambda s: int((s == 0).sum())),
    "out of school % (unweighted)": g[TARGET].apply(lambda s: (1 - s.mean()) * 100),
    "out of school % (weighted)": g[[TARGET, "weights"]].apply(weighted_share_out),
}).loc[ORDER].round(1)
sample.to_csv(OUT / "sample_and_prevalence.csv")
sample

,children,households,PSUs,out of school (n),out of school % (unweighted),out of school % (weighted)
district,,,,,,
Karachi Central,2917,1462,117,636,21.8,20.5
Islamabad,1696,771,50,179,10.6,10.1
Lahore,7956,3524,225,1288,16.2,16.4
Quetta,3259,1113,62,1134,34.8,34.8
Peshawar,6315,2037,90,1732,27.4,25.0
Kohistan,2604,943,54,1653,63.5,60.3
Dera Bugti,1270,455,19,995,78.3,68.4
Rajanpur,2833,1012,44,1380,48.7,48.0
Tharparkar,1742,675,36,1003,57.6,56.1


## Step 3 · Attendance by group
Survey-weighted % of children attending school.

In [3]:
data["age group"] = pd.cut(data["age"], [4, 9, 12, 16], labels=["5-9", "10-12", "13-16"])
rows = {}
for name, col in [("gender", "gender"), ("age", "age group"), ("region", "region")]:
    for (district, level), d in data.groupby(["district", col], observed=True):
        rows[(district, f"{name}: {level}")] = np.average(d[TARGET], weights=d["weights"]) * 100
by_group = pd.Series(rows).unstack().loc[ORDER].round(1)
by_group.to_csv(OUT / "attendance_by_group.csv")
by_group

,age: 10-12,age: 13-16,age: 5-9,gender: female,gender: male,region: rural,region: urban
Karachi Central,84.8,72.2,81.4,79.9,79.1,NaN,79.5
Islamabad,92.1,86.5,91.1,89.0,90.7,90.5,89.3
Lahore,87.9,77.7,85.2,85.1,82.4,NaN,83.6
Quetta,63.9,55.2,72.4,55.3,72.8,63.6,67.7
Peshawar,85.1,66.4,75.3,64.4,84.1,70.8,80.5
Kohistan,53.7,33.9,35.9,10.5,66.9,39.7,NaN
Dera Bugti,35.9,11.9,36.7,23.8,36.7,27.7,43.0
Rajanpur,60.4,44.0,51.8,45.3,57.4,46.4,82.5
Tharparkar,51.5,28.5,47.1,32.3,53.3,43.0,56.1


## Step 4 · Missing values
% of children without a value, per predictor and district. In notebook 03 a predictor is not used in a training set where it is missing for more than 50 % of children.

In [4]:
missing = data.groupby("district")[FEATURES].apply(lambda d: d.isna().mean() * 100).T[ORDER].round(1)
missing.to_csv(OUT / "missing_values.csv")
missing.sort_values(ORDER, ascending=False)

district,Karachi Central,Islamabad,Lahore,Quetta,Peshawar,Kohistan,Dera Bugti,Rajanpur,Tharparkar
property_owner_gender,99.7,91.6,95.8,97.1,83.8,30.8,34.5,63.0,71.0
monthly_income,98.7,99.0,97.9,98.4,97.8,99.6,99.6,98.2,98.0
can_report_income,98.6,99.0,97.9,98.1,97.8,99.6,99.6,97.8,97.9
work_days_last_month,98.5,98.9,97.6,97.4,97.3,93.1,68.0,91.8,97.4
employment_status,98.5,98.9,97.6,97.4,97.3,93.1,68.0,91.8,97.4
born_district_type,97.1,82.6,93.7,95.4,92.7,100.0,100.0,99.6,100.0
migration_reason,97.1,82.6,93.7,95.4,92.7,100.0,100.0,99.6,100.0
transport_mode,94.0,94.8,91.6,90.1,60.3,50.5,5.0,24.8,53.0
years_to_complete_primary,92.9,95.5,95.0,94.7,94.3,96.8,99.5,95.4,96.7
computer_location,88.8,86.9,93.6,92.9,93.3,99.3,99.7,100.0,99.9


## Step 5 · Association with attendance
Numeric predictors: absolute correlation with attendance (point-biserial |r|). Categorical predictors: Cramér's V. Children with an observed value only.

In [5]:
def association(x, y):
    ok = x.notna()
    if ok.sum() < 30 or x[ok].nunique() < 2 or y[ok].nunique() < 2:
        return np.nan
    if pd.api.types.is_numeric_dtype(x):
        return abs(np.corrcoef(x[ok].astype(float), y[ok])[0, 1])
    table = pd.crosstab(x[ok], y[ok])
    if table.shape[1] < 2:
        return np.nan
    chi2 = chi2_contingency(table, correction=False)[0]
    return np.sqrt(chi2 / (table.values.sum() * (min(table.shape) - 1)))

assoc = pd.DataFrame({d: {f: association(s[f], s[TARGET]) for f in FEATURES}
                      for d, s in data.groupby("district")})[ORDER].round(3)
assoc.to_csv(OUT / "association_with_attendance.csv")
pd.DataFrame({d: [FEATURE_NAMES[f] for f in assoc[d].nlargest(10).index] for d in ORDER},
             index=range(1, 11))

,Karachi Central,Islamabad,Lahore,Quetta,Peshawar,Kohistan,Dera Bugti,Rajanpur,Tharparkar
1,Work days last month,Can do math,Employment status,Can do math,Can do math,Can do math,Can do math,Can do math,Can do math
2,Head education level,Worked last month,Worked last month,Migration reason,Migration reason,Gender,Worked last month,Worked last month,Head education level
3,Can do math,Head education level,Income used for household,Employment status,Worked last month,Worked last month,No-mobile reason,Toilet type,Can write
4,Worked last month,Income used for household,Can report income,Toilet type,Income used for household,Handwashing water source,Cooking water source,Head education level,No-mobile reason
5,Income used for household,Has job,Can do math,Transport mode,Computer location,Cooking water source,Property owner gender,Shared toilet,Gender
6,Migration reason,Toilet type,Head education level,Can report income,Employment status,Head education level,Heating fuel,Handwashing water source,Toilet type
7,Employment status,Migration reason,Hungry but did not eat,Can write,Gender,Work days last month,Handwashing water source,Connected to sewerage,Shared toilet
8,Has internet,Occupancy status,Has job,Worked last month,Monthly income,Employment status,Toilet type,Can report income,Worked last month
9,Born-district type,Cooking water source,Household ran out of food,Head education level,Head education level,Heating fuel,Head education level,Cooking water source,Used mobile
10,Used mobile,Has internet,Worried about food,Income used for household,Work days last month,Transport mode,Total assets,Heating fuel,Income used for household


## Step 6 · School-dependent predictors
Can do math, can write and years to complete primary are recorded only for children who attend or attended school, so they partly encode the outcome. They are excluded from the models and used only in a sensitivity analysis.

In [6]:
rows = []
for f in SCHOOL_DEPENDENT:
    for d, s in data.groupby("district"):
        recorded = s[f].notna()
        rows.append({"predictor": FEATURE_NAMES[f], "district": d,
                     "recorded % (attending)": recorded[s[TARGET] == 1].mean() * 100,
                     "recorded % (out of school)": recorded[s[TARGET] == 0].mean() * 100,
                     "association": assoc.loc[f, d]})
school = pd.DataFrame(rows)
school.to_csv(OUT / "school_dependent_predictors.csv", index=False)
school.groupby("predictor")[["recorded % (attending)", "recorded % (out of school)",
                             "association"]].mean().round(2)

,recorded % (attending),recorded % (out of school),association
predictor,,,
Can do math,50.49,53.62,0.45
Can write,45.84,17.81,0.12
Years to complete primary,0.00,18.15,NaN


## Step 7 · Table D1: predictors used
‡ = used only in the districts where it is recorded for at least 50 % of children.

In [7]:
used_in = {f: [d for d in ORDER if missing.loc[f, d] <= MAX_MISSING_SHARE * 100] for f in FEATURES}
long_rows, wide_rows = [], []
for domain, variables in FEATURE_DOMAINS.items():
    shown = []
    for f in variables:
        if f in SCHOOL_DEPENDENT:
            status = "not used: school-dependent"
        elif not used_in[f]:
            status = "not used: > 50 % missing in every district"
        elif len(used_in[f]) == len(ORDER):
            status = "used in all districts"
            shown.append(FEATURE_NAMES[f])
        else:
            status = "used in " + ", ".join(used_in[f])
            shown.append(FEATURE_NAMES[f] + "‡")
        long_rows.append({"Domain": domain, "Predictor": FEATURE_NAMES[f],
                          "Level": FEATURE_LEVEL[f], "Status": status})
    wide_rows.append({"Domain": domain, "PSLM features used": "; ".join(shown) or "—"})
pd.DataFrame(long_rows).to_csv(TABLES / "tableD1_predictors_long.csv", index=False)
table_d1 = pd.DataFrame(wide_rows)
table_d1.to_csv(TABLES / "tableD1_predictors.csv", index=False)
table_d1

,Domain,PSLM features used
0,Child demographics,Age; Gender; Marital status; Relationship to head; Residence status; Born in...
1,Education & skills,—
2,Household head,Head age; Head gender; Head education level; Reason for headship
3,Child work & income,Worked last month‡; Has job‡; Received in-kind wages‡; Income used for house...
4,Household economic status,Household income (log); Total assets; House owner gender; Occupancy status; ...
5,Food security,Worried about food; Household ran out of food; Hungry but did not eat
6,"Water, sanitation & hygiene",Toilet type; Shared toilet; Connected to sewerage; Has handwashing place; Ha...
7,Housing & location,Dwelling type; Number of rooms; Heating fuel; Region (urban/rural); Transpor...
8,Digital access,Has internet; Has computer; Has laptop; Household has mobile phone; Child ha...
9,Health & disability,Disability; Sick in last 2 weeks


## Step 8 · Subgroups: age, sex, disability (Table 4)
Unweighted % attending. Disability = 'some difficulty' or worse in any Washington Group domain. Table 4 pools the nine districts; 95 % CIs of the gap from a bootstrap of households (2,000 resamples).

In [8]:
pct = lambda s: s.mean() * 100
by_age = data.groupby(pd.cut(data["age"], [4, 6, 9, 12, 14, 16],
                             labels=["5-6", "7-9", "10-12", "13-14", "15-16"]), observed=True)[TARGET]
by_sex = data.pivot_table(index="district", columns="gender", values=TARGET, aggfunc=pct).loc[ORDER]
disabled = data["disability"].astype(str) == "yes"
by_disability = data.assign(disabled=disabled).pivot_table(
    index="district", columns="disabled", values=TARGET, aggfunc=["size", pct]).loc[ORDER]
by_disability.columns = ["children without", "children with", "% attending without", "% attending with"]
by_age.agg(pct).round(1).to_csv(OUT / "attendance_by_age.csv")
by_sex.round(1).to_csv(OUT / "attendance_by_sex.csv")
by_disability.round(1).to_csv(OUT / "attendance_by_disability.csv")
display(by_age.agg(pct).round(1).to_frame("% attending").T, by_sex.round(1), by_disability.round(1))

age,5-6,7-9,10-12,13-14,15-16
% attending,54.9,76.2,74.2,67.1,56.5


gender,female,male
district,,
Karachi Central,78.1,78.2
Islamabad,88.5,90.3
Lahore,84.9,82.8
Quetta,55.4,72.5
Peshawar,61.8,82.0
Kohistan,7.9,61.5
Dera Bugti,14.1,26.8
Rajanpur,44.5,56.8
Tharparkar,30.8,51.8


,children without,children with,% attending without,% attending with
district,,,,
Karachi Central,2789,128,78.5,72.7
Islamabad,1647,49,90.1,67.3
Lahore,7676,280,84.7,60.7
Quetta,3142,117,65.6,55.6
Peshawar,6115,200,73.1,57.0
Kohistan,2576,28,36.7,17.9
Dera Bugti,1264,6,21.7,16.7
Rajanpur,2697,136,52.2,33.1
Tharparkar,1682,60,42.6,36.7


In [9]:
HEAD_EDU = pd.cut(data["head_edu_level_num"], [-1, 0, 3, 4],
                  labels=["No schooling", "School (classes 1-12)", "Degree"])

def gap_ci(d, n_boot=2000, seed=42):
    """Attendance gap (without minus with difficulty), percentage points, household bootstrap."""
    y, dis = d[TARGET].to_numpy(), (d["disability"].astype(str) == "yes").to_numpy()
    rows_of = d.reset_index(drop=True).groupby("hhcode").indices
    households = list(rows_of)
    rng = np.random.default_rng(seed)
    boot = []
    for _ in range(n_boot):
        idx = np.concatenate([rows_of[h] for h in rng.choice(households, len(households))])
        boot.append(y[idx][~dis[idx]].mean() - y[idx][dis[idx]].mean())
    gap = y[~dis].mean() - y[dis].mean()
    return gap * 100, *(np.percentile(boot, [2.5, 97.5]) * 100)

rows = []
for level, d in [*data.groupby(HEAD_EDU, observed=True), ("All children", data)]:
    dis = d["disability"].astype(str) == "yes"
    gap, lo, hi = gap_ci(d)
    rows.append({"Education of household head": level,
                 "Without difficulty: children": (~dis).sum(),
                 "Without difficulty: attending (%)": pct(d.loc[~dis, TARGET]),
                 "With difficulty: children": dis.sum(),
                 "With difficulty: attending (%)": pct(d.loc[dis, TARGET]),
                 "Gap (pp)": gap, "Gap 95% CI low": lo, "Gap 95% CI high": hi})
table4 = pd.DataFrame(rows).round(1)
table4.to_csv(TABLES / "table4_disability_by_head_education.csv", index=False)
table4

,Education of household head,Without difficulty: children,Without difficulty: attending (%),With difficulty: children,With difficulty: attending (%),Gap (pp),Gap 95% CI low,Gap 95% CI high
0,No schooling,13676,51.2,475,42.7,8.4,3.8,12.9
1,School (classes 1-12),12761,79.9,454,62.6,17.4,12.3,22.1
2,Degree,2965,91.3,72,80.6,10.7,1.5,19.9
3,All children,29588,67.7,1004,54.6,13.2,10.0,16.4
